In [ ]:
%pip install requests
%pip install pandas
%pip install beautifulsoup4
%pip install boto3
%pip install logging

## 동아일보

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import os
import re
from datetime import datetime
from urllib.parse import urlparse

logging.basicConfig(level='DEBUG')

header = {'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36'),}

output_dir = './donga'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

categories = {
    '정치': 'https://www.donga.com/news/Politics',
    '경제': 'https://www.donga.com/news/Economy',
    '국제': 'https://www.donga.com/news/Inter',
    '사회_사건': 'https://www.donga.com/news/Society/Event', # 앞 분기 기준
    '경제_과학': 'https://www.donga.com/news/It/List', # 앞 분기 기준
    '사회': 'https://www.donga.com/news/Society',
    '스포츠': 'https://www.donga.com/news/Sports',
}

result_categories = {
    '정치': 'politics',
    '경제': 'economy',
    '국제': 'world',
    '사회_사건': 'accident',
    '경제_과학': 'science',
    '사회': 'society',
    '스포츠': 'sports',
}

# 현재 날짜의 오전 8시 (시간대는 +09:00 기준)
today = datetime.now().replace(hour=8, minute=0, second=0, microsecond=0, tzinfo=datetime.now().astimezone().tzinfo)

def scrap_article(article_url, category):
    response = requests.get(article_url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')

    try:
        page_category = soup.find('ol', class_='breadcrumb').find('li', class_='breadcrumb_item').find('a').get_text(strip=True)
        logging.info(f"page_category: {page_category}")
        
        logging.info(f"category: {category}")

        category_parts = category.split('_')
        
        # 카테고리가 아닌 글은 패스
        if page_category not in category_parts:
            logging.info(f"{article_url} 는 해당 카테고리가 아니어서 생략합니다. ")
            return None
    except AttributeError:
        logging.error(f"{article_url} 는 찾을 수 없습니다.")
        return None

    # 헤드라인 스크랩
    try:
        title = soup.find('section', {'class': 'head_group'}).find('h1').get_text(strip=True)
    except AttributeError:
        title = 'N/A'
        
    try:
        date_button = soup.find('button', {'aria-controls': 'dateInfo'})
        time_span = date_button.find('span', {'aria-hidden': 'true'})
        time = time_span.get_text(strip=True)  # "2024-09-30 21:44" 형태
        time = time.replace(" ", "T") + ":00+09:00"  # ISO 8601 형식으로 변환 (예: 2024-09-30T21:44:00+09:00)
    except AttributeError:
        time = 'N/A'

    # 본문 스크랩
    try:
        # 광고, 이미지 등 제외하고 텍스트만 추출
        content_section = soup.find('section', class_='news_view')
        for tag in content_section(['figure', 'script', 'div', 'ul']):  # 불필요한 태그 제거
            tag.decompose()
        content = content_section.get_text(separator="\n", strip=True)
    except AttributeError:
        content = 'N/A'

    result_category = result_categories.get(category, 'unknown')

    return {'title': title, 'source_url': article_url, 'content': content, 'category': result_category, 'source_created_at': time, 'press_id': 2, 'press_name' : '동아일보'}

# 파일 생성
def save_article_as_json(article_data, date, file_idx):
    category = article_data.get('category')
    file_name = date + "-" + str(file_idx) + '.json'
    result_dir = output_dir + "/" + category + "/" + date + "-" + str(file_idx)
    if not os.path.exists(result_dir):
        os.makedirs(result_dir)

    file_path = os.path.join(result_dir, 'source-' + file_name)
    
    # 단일 dataframe 생성
    single_article_df = pd.DataFrame([article_data])
    
    # json으로 변환
    single_article_df.to_json(file_path, orient='records', lines=True, force_ascii=False)
    logging.info(f"Article saved to {file_path}")

def check_for_new_articles(base_url, category):
    response = requests.get(base_url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')

    article_count = 0
    cards = soup.find_all('article', class_='news_card')
    if category not in ['사회_사건', '경제_과학']:
        cards.reverse()
    for card in cards:
        if article_count >= 4:
            break

        link_tag = card.find('a', href=True)
        if link_tag and 'href' in link_tag.attrs:
            article_link = link_tag['href']
        else:
            logging.warning("article_link is None or missing 'href'.")
            continue
    
        logging.debug(f"Scraping article: {article_link}")

        result = scrap_article(article_link, category)

        if result is None:
            logging.error("result is None");
            continue

        # if result.get('category') not in ['accident', 'science']:
        # 'source_created_at'를 datetime 객체로 변환
        article_time = datetime.strptime(result.get('source_created_at'), '%Y-%m-%dT%H:%M:%S%z')
        # 오늘 오전 8시 이후가 아니면 continue
        if article_time < today:
            logging.info(f"{article_link} 은 오늘 오전 8시 이전 기사이므로 생략합니다.")
            continue
        
        print(result)
        article_count += 1
        date = re.match(r'^[^T]+', result.get('source_created_at')).group()
        save_article_as_json(result, date, article_count)

def scrap_all_categories():
    
    for category, base_url in categories.items():
        logging.info(f"category (base_url): {category} ({base_url})")
        check_for_new_articles(base_url, category)
        time.sleep(2)

scrap_all_categories()

## 중앙일보

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import os
import re
from datetime import datetime
from urllib.parse import urlparse

logging.basicConfig(level='DEBUG')

header = {'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36'),}

output_dir = './joongang'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

categories = {
    '정치': 'https://www.joongang.co.kr/politics',
    '경제': 'https://www.joongang.co.kr/money',
    '국제': 'https://www.joongang.co.kr/world',
    '사회_사건': 'https://www.joongang.co.kr/society/accident', # 앞 분기 기준
    '경제_과학': 'https://www.joongang.co.kr/money/science', # 앞 분기 기준
    '사회': 'https://www.joongang.co.kr/society',
    '스포츠': 'https://www.joongang.co.kr/sports',
}


result_categories = {
    '정치': 'politics',
    '경제': 'economy',
    '국제': 'world',
    '사회_사건': 'accident',
    '경제_과학': 'science',
    '사회': 'society',
    '스포츠': 'sports',
}

# 현재 날짜의 오전 8시 (시간대는 +09:00 기준)
today = datetime.now().replace(hour=8, minute=0, second=0, microsecond=0, tzinfo=datetime.now().astimezone().tzinfo)

def scrap_article(article_url, category):
    response = requests.get(article_url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')

    try:

        page_category = soup.find('a', {'class': 'title'}).get_text(strip=True)
        logging.info(f"page_category: {page_category}")
        
        logging.info(f"category: {category}")

        category_parts = category.split('_')
        
        # 카테고리가 아닌 글은 패스
        if page_category not in category_parts:
            logging.info(f"{article_url} 는 해당 카테고리가 아니어서 생략합니다. ")
            return None
    except AttributeError:
        logging.error(f"{article_url} 는 찾을 수 없습니다.")
        return None

    # 헤드라인 스크랩
    try:
        title = soup.find('h1', {'class': 'headline'}).get_text(strip=True)
    except AttributeError:
        title = 'N/A'

    # 생성일 스크랩
    try:
        date = soup.find('p', {'class': 'date'})
        time = date.find('time').get('datetime')
    except AttributeError:
        time = 'N/A'

    # 본문 스크랩
    try:
        content = soup.find('div', {'class': 'article_body'}).get_text(strip=True)
    except AttributeError:
        content = 'N/A'

    result_category = result_categories.get(category, 'unknown')

    return {'title': title, 'source_url': article_url, 'content': content, 'category': result_category, 'source_created_at': time, 'press_id': 1, 'press_name': '중앙일보'}

# 파일 생성
def save_article_as_json(article_data, date, file_idx):
    category = article_data.get('category')
    file_name = date + "-" + str(file_idx) + '.json'
    result_dir = output_dir + "/" + category + "/" + date + "-" + str(file_idx)
    if not os.path.exists(result_dir):
        os.makedirs(result_dir)

    file_path = os.path.join(result_dir, 'source-' + file_name)
    
    # 단일 dataframe 생성
    single_article_df = pd.DataFrame([article_data])
    
    # json으로 변환
    single_article_df.to_json(file_path, orient='records', lines=True, force_ascii=False)
    logging.info(f"Article saved to {file_path}")

def check_for_new_articles(base_url, category):
    response = requests.get(base_url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')

    article_count = 0
    cards = soup.find_all('li', class_='card')
    if category not in ['사회_사건', '경제_과학']:
        cards.reverse()
    for card in cards:
        if article_count >= 4:
            break

        link_tag = card.find('a', href=True)
        if link_tag and 'href' in link_tag.attrs:
            article_link = link_tag['href']
        else:
            logging.warning("article_link is None or missing 'href'.")
            continue
    
        logging.debug(f"Scraping article: {article_link}")

        result = scrap_article(article_link, category)

        if result is None:
            continue

        if result.get('category') not in ['accident', 'science']:
            # 'source_created_at'를 datetime 객체로 변환
            article_time = datetime.strptime(result.get('source_created_at'), '%Y-%m-%dT%H:%M:%S%z')
            # 오늘 오전 8시 이후가 아니면 continue
            if article_time < today:
                logging.info(f"{article_link} 은 오늘 오전 8시 이전 기사이므로 생략합니다.")
                continue

        print(result)
        article_count += 1
        date = re.match(r'^[^T]+', result.get('source_created_at')).group()
        save_article_as_json(result, date, article_count)

def scrap_all_categories():
    
    for category, base_url in categories.items():
        logging.info(f"category (base_url): {category} ({base_url})")
        check_for_new_articles(base_url, category)
        time.sleep(2)

scrap_all_categories()

## 중앙일보 단일 기사 스크랩

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import logging
import os
import re

logging.basicConfig(level='DEBUG')

header = {'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36'),}

output_dir = './joongang'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def scrap_article(article_url):
    response = requests.get(article_url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')

    # 헤드라인 스크랩
    try:
        title = soup.find('h1', {'class': 'headline'}).get_text(strip=True)
    except AttributeError:
        title = 'N/A'

    # 생성일 스크랩
    try:
        date = soup.find('p', {'class': 'date'})
        time = date.find('time').get('datetime')
    except AttributeError:
        time = 'N/A'

    # 본문 스크랩
    try:
        content = soup.find('div', {'class': 'article_body'}).get_text(strip=True)
    except AttributeError:
        content = 'N/A'

    return {
        'title': title,
        'source_url': article_url,
        'content': content,
        'category': 'sports',  # 이 부분은 해당 카테고리에 맞게 수정할 수 있습니다.
        'source_created_at': time,
        'press_id': 1,
        'press_name': '중앙일보'
    }

# 파일 생성
def save_article_as_json(article_data, date, file_idx):
    category = "sports"
    file_name = date + "-" + str(file_idx) + '.json'
    result_dir = output_dir + "/" + category + "/" + date + "-" + str(file_idx)
    if not os.path.exists(result_dir):
        os.makedirs(result_dir)

    file_path = os.path.join(result_dir, 'source-' + file_name)
    
    # 단일 dataframe 생성
    single_article_df = pd.DataFrame([article_data])
    
    # json으로 변환
    single_article_df.to_json(file_path, orient='records', lines=True, force_ascii=False)
    logging.info(f"Article saved to {file_path}")

def scrap_single_categories():
    article_url = "https://www.joongang.co.kr/article/25284548"
    
    result = scrap_article(article_url)
    
    if result:
        date = re.match(r'^[^T]+', result.get('source_created_at')).group()  # 날짜 추출
        save_article_as_json(result, date, 1)
    else:
        logging.info(f"Failed to scrape article from {article_url}")

scrap_single_categories()

## 동아일보 단일 기사 스크랩

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import os
import re
from datetime import datetime
from urllib.parse import urlparse

logging.basicConfig(level='DEBUG')

header = {'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36'),}

output_dir = './donga'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def scrap_article(article_url):
    response = requests.get(article_url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')

    # 헤드라인 스크랩
    try:
        title = soup.find('section', {'class': 'head_group'}).find('h1').get_text(strip=True)
    except AttributeError:
        title = 'N/A'
        
    try:
        date_button = soup.find('button', {'aria-controls': 'dateInfo'})
        time_span = date_button.find('span', {'aria-hidden': 'true'})
        time = time_span.get_text(strip=True)  # "2024-09-30 21:44" 형태
        time = time.replace(" ", "T") + ":00+09:00"  # ISO 8601 형식으로 변환 (예: 2024-09-30T21:44:00+09:00)
    except AttributeError:
        time = 'N/A'

    # 본문 스크랩
    try:
        # 광고, 이미지 등 제외하고 텍스트만 추출
        content_section = soup.find('section', class_='news_view')
        for tag in content_section(['figure', 'script', 'div', 'ul']):  # 불필요한 태그 제거
            tag.decompose()
        content = content_section.get_text(separator="\n", strip=True)
    except AttributeError:
        content = 'N/A'

    return {
        'title': title,
        'source_url': article_url,
        'content': content,
        'category': 'society',  # 이 부분은 해당 카테고리에 맞게 수정할 수 있습니다.
        'source_created_at': time,
        'press_id': 2,
        'press_name': '동아일보'
    }

# 파일 생성
def save_article_as_json(article_data, date, file_idx):
    category = "society"
    file_name = date + "-" + str(file_idx) + '.json'
    result_dir = output_dir + "/" + category + "/" + date + "-" + str(file_idx)
    if not os.path.exists(result_dir):
        os.makedirs(result_dir)

    file_path = os.path.join(result_dir, 'source-' + file_name)
    
    # 단일 dataframe 생성
    single_article_df = pd.DataFrame([article_data])
    
    # json으로 변환
    single_article_df.to_json(file_path, orient='records', lines=True, force_ascii=False)
    logging.info(f"Article saved to {file_path}")

def scrap_single_categories():
    article_url = "https://www.donga.com/news/Society/article/all/20241016/130226096/1"
    
    result = scrap_article(article_url)
    
    if result:
        date = re.match(r'^[^T]+', result.get('source_created_at')).group()  # 날짜 추출
        save_article_as_json(result, date, 2)
    else:
        logging.info(f"Failed to scrape article from {article_url}")

scrap_single_categories()